# Summary Figure: Reconstruction Error, Target Linearity, and Decoder Outputs


In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from matplotlib.lines import Line2D
from src.model import VAE
from src.data import get_mnist, get_cifar10
from src.utils import load_json

# ── Config ────────────────────────────────────────────────────────────────────
DATASET        = 'mnist'            # 'mnist' | 'cifar10'
VIS_BETAS      = [0.05, 0.1, 1.0, 10.0]  # betas shown in the decoder-output panel
SAMPLE_CLASSES = list(range(8))     # one image per class; shown as columns
IMG_IDX        = 0                  # which image to pick from each class

DATA_DICT = {
    'mnist':   (1, 28, get_mnist),
    'cifar10': (3, 32, get_cifar10),
}
DATA_CONFIG  = DATA_DICT[DATASET]
IN_CHANNELS  = DATA_CONFIG[0]
INPUT_SIZE   = DATA_CONFIG[1]
HIDDEN_DIM   = 512
LATENT_DIM   = 32
NUM_HIDDEN   = 2

PARAMS_ROOT  = ROOT / 'params' / 'vae' / DATASET
RESULTS_ROOT = ROOT / 'result' / 'vae' / DATASET
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dataset: {DATASET}  |  Device: {DEVICE}')


In [ ]:
# ── Pick one test image per class ────────────────────────────────────────────
_, testset = DATA_CONFIG[2]()
targets    = torch.tensor(testset.targets)

sample_imgs = []
for cls in SAMPLE_CLASSES:
    idx = (targets == cls).nonzero(as_tuple=True)[0][IMG_IDX].item()
    sample_imgs.append(testset[idx][0])
# shape: (N_IMGS, C, H, W)
sample_imgs = torch.stack(sample_imgs).to(DEVICE)


def load_vae(ckpt_path):
    model = VAE(
        in_channels=IN_CHANNELS, input_size=INPUT_SIZE,
        hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM, num_hidden=NUM_HIDDEN,
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    return model


@torch.no_grad()
def reconstruct(model, imgs):
    """Return reconstructed images (N, C, H, W) on CPU."""
    recon, _, _ = model(imgs)
    return recon.view_as(imgs).cpu()


def to_img(t):
    t = t.squeeze()
    if t.ndim == 3:          # RGB
        t = t.permute(1, 2, 0).clamp(0, 1)
    return t.cpu().numpy()


def show_img(ax, t):
    img  = to_img(t)
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
    ax.axis('off')


# ── Load metrics and reconstructions for every available β ────────────────────
beta_dirs = sorted(PARAMS_ROOT.iterdir(), key=lambda p: float(p.name))
all_betas = [float(p.name) for p in beta_dirs]

beta_data = {}
for beta_dir in beta_dirs:
    beta = float(beta_dir.name)

    best_ckpts = list(beta_dir.glob('best_*.pt'))
    if not best_ckpts:
        print(f'  β={beta}: no best checkpoint, skipping')
        continue
    best_ckpt  = best_ckpts[0]
    best_epoch = int(best_ckpt.stem.split('_')[1])   # 'best_10' → 10

    result_dir  = RESULTS_ROOT / beta_dir.name
    loss_data   = load_json(result_dir / 'loss.json')
    tl_data     = load_json(result_dir / 'tl.json')

    epochs      = [d['epoch']       for d in loss_data]
    train_recon = [d['train_recon'] for d in loss_data]
    eval_recon  = [d['eval_recon']  for d in loss_data]
    tl_vals     = tl_data['target_linearity']

    recons = None
    if beta in VIS_BETAS:
        model  = load_vae(best_ckpt)
        recons = reconstruct(model, sample_imgs)   # (N_IMGS, C, H, W)

    beta_data[beta] = dict(
        epochs=epochs, train_recon=train_recon, eval_recon=eval_recon,
        tl_vals=tl_vals, best_epoch=best_epoch, recons=recons,
    )
    print(f'  β={beta:5.2f}  best_epoch={best_epoch}  TL={tl_vals[best_epoch-1]:.4f}')

print('Done.')


In [ ]:
# ── Figure layout ─────────────────────────────────────────────────────────────
#   Col 0  : train/val reconstruction error  (line plot)
#   Col 1  : target linearity                (line plot)
#   Col 2  : decoder output grid             (image grid)
#              Row 0 : original test images
#              Row 1+: VAE reconstruction for each selected β

n_imgs     = len(SAMPLE_CLASSES)
n_img_rows = len(VIS_BETAS) + 1   # original + one row per selected β
colors     = plt.cm.viridis(np.linspace(0, 1, len(all_betas)))
beta_color = {b: c for b, c in zip(sorted(all_betas), colors)}

fig = plt.figure(figsize=(15, 3))
outer = gridspec.GridSpec(
    1, 3, figure=fig,
    width_ratios=[1, 1, 1.5],
    wspace=0.22,
)

ax_recon = fig.add_subplot(outer[0])
ax_tl    = fig.add_subplot(outer[1])
inner    = gridspec.GridSpecFromSubplotSpec(
    n_img_rows, n_imgs + 1,       # +1 col for row labels
    subplot_spec=outer[2],
    hspace=0.06, wspace=0.04,
    width_ratios=[0.7] + [1] * n_imgs,
)

# ── Panel 1: reconstruction error ─────────────────────────────────────────────
for beta, d in sorted(beta_data.items()):
    c = beta_color[beta]
    ax_recon.plot(d['epochs'], d['train_recon'], color=c, linestyle='--', linewidth=2.0, alpha=0.7)
    ax_recon.plot(d['epochs'], d['eval_recon'],  color=c, linestyle='-',  linewidth=2.5, label=f'β={beta}')

legend_handles, legend_labels = ax_recon.get_legend_handles_labels()
style_handles = [
    Line2D([0], [0], color='gray', linestyle='--', linewidth=2.0, label='train'),
    Line2D([0], [0], color='gray', linestyle='-',  linewidth=2.5, label='val'),
]
ax_recon.legend(
    handles=legend_handles + style_handles,
    labels=legend_labels + ['train', 'val'],
    fontsize=10, ncol=2, loc='upper right',
)
ax_recon.set_xlabel('Epoch', fontsize=15, fontweight='bold')
ax_recon.set_ylabel('Reconstruction Error', fontsize=15, fontweight='bold')

# ── Panel 2: target linearity ──────────────────────────────────────────────────
for beta, d in sorted(beta_data.items()):
    ax_tl.plot(d['epochs'], d['tl_vals'], color=beta_color[beta], linewidth=2.5, label=f'β={beta}')

# ax_tl.legend(fontsize=10, ncol=2, loc='lower right')
ax_tl.set_xlabel('Epoch', fontsize=15, fontweight='bold')
ax_tl.set_ylabel('Target Linearity', fontsize=15, fontweight='bold')

# ── Panel 3: image grid ────────────────────────────────────────────────────────
# Row 0: original images
ax_lbl = fig.add_subplot(inner[0, 0])
ax_lbl.axis('off')
ax_lbl.text(0.95, 0.5, 'original', ha='right', va='center', fontsize=8,
            transform=ax_lbl.transAxes, fontweight='bold')

for col, img in enumerate(sample_imgs):
    ax = fig.add_subplot(inner[0, col + 1])
    show_img(ax, img)
    # ax.set_title(f'cls {SAMPLE_CLASSES[col]}', fontsize=6, pad=2)

# Rows 1+: reconstructions per selected β
for row, beta in enumerate(VIS_BETAS):
    d = beta_data.get(beta)
    ax_lbl = fig.add_subplot(inner[row + 1, 0])
    ax_lbl.axis('off')
    ax_lbl.text(0.95, 0.5, f'β={beta}', ha='right', va='center', fontsize=8,
                transform=ax_lbl.transAxes, fontweight='bold')

    for col in range(n_imgs):
        ax = fig.add_subplot(inner[row + 1, col + 1])
        if d is not None and d['recons'] is not None:
            show_img(ax, d['recons'][col])
        else:
            ax.axis('off')

# fig.suptitle(
#     f'VAE on {DATASET.upper()} Generation',
#     fontsize=20, fontweight='bold', y=1.00,
# )

out_path = ROOT / 'plot' / f'vae_{DATASET}.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.tight_layout()
plt.show()
print(f'Saved → {out_path.relative_to(ROOT)}')
